# Import Libraries

In [1]:
%pip install torch torchvision tensorboard

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
from torchvision import transforms as T
from utils.set_seed import set_random_seed
from torch.utils.tensorboard import SummaryWriter
import pandas as pd

device = 'cpu'
if torch.backends.mps.is_available(): device = torch.device('mps')
elif torch.cuda.is_available(): device = torch.device('cuda')

In [4]:
print(device)

cuda


In [5]:
from weathernet import WeatherNet
from weathernetplusplus import WeatherNetPlusPlus
from mtl_weathernet import MtlWeatherNet
from weathernet_transformer import WeatherNetTransformer

from data.data import get_dataloaders
import utils.trainer as trainer

# Set Hyperparameters, Load Dataset

In [6]:
# Hyperparameters
# TODO: experiment with different hyperparameters
batch_size = 192
learning_rate = 0.001

set_random_seed(42) # seed for reproducibility

In [7]:
# Load BDD100KPlus dataset
trainloader, valloader, testloader = get_dataloaders(dataset_name="Bdd100kPlus", batch_size=batch_size)

In [8]:
# Print dataset statistics
print(f"Number of training samples: {len(trainloader.dataset)}")
print(f"Number of validation samples: {len(valloader.dataset)}")
print(f"Number of test samples: {len(testloader.dataset)}")

# print batches
print(f"Number of batches in training set: {len(trainloader)}")
print(f"Number of batches in validation set: {len(valloader)}")
print(f"Number of batches in test set: {len(testloader)}")

Number of training samples: 70000
Number of validation samples: 10000
Number of test samples: 20000
Number of batches in training set: 365
Number of batches in validation set: 53
Number of batches in test set: 105


# Initialize WeatherNet Model

In [9]:
# model: WeatherNet = WeatherNet()
# model: WeatherNetPlusPlus = WeatherNetPlusPlus()
model: MtlWeatherNet = MtlWeatherNet()
# model: WeatherNetTransformer = WeatherNetTransformer()

# Print the model architecture
print(model)

# Define optimizer for the model
# Could explore SGD, Adam, AdamW, etc
optimizer = torch.optim.Adam(model.parameters())

MtlWeatherNet(
  (backbone_model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequentia

In [10]:
# saved_state = './checkpoints/mtl_8.pth' # path to saved model state
saved_state = None
if saved_state is not None:
    print(f"Loading saved state from {saved_state}")
    model.load_checkpoint(saved_state)

In [11]:
import os
# trainloader.num_workers = int(os.cpu_count())
# valloader.num_workers = int(os.cpu_count())
# testloader.num_workers = int(os.cpu_count())

print(f"Number of workers: {trainloader.num_workers}")
print(f"Number of workers: {valloader.num_workers}")
print(f"Number of workers: {testloader.num_workers}")

Number of workers: 8
Number of workers: 8
Number of workers: 8


# Model Training

In [12]:
# Train the WeatherNet model and record losses
# Use the train function to train the model with optimizer on the trainloader for a specified number of epochs (e.g., 5)
# Record the training and test losses in wn_train_losses and wn_test_losses respectively, pass hyperparameters

# logs to runs/WeatherNet/ or runs/WeatherNetPlusPlus/ or runs/MtlWeatherNet/
writer = SummaryWriter(log_dir=f"runs/{model.name}")

epochs=20
trainer.train(model, optimizer, trainloader, valloader, epochs, device, "./checkpoints", writer=writer)

writer.flush()

Begin training
Epoch 1/20
* Batch 10/365 - fog loss: 0.07 | glare loss: 0.32 | road loss: 0.58 | traffic loss: 0.26 | weather loss: 1.00 | scene loss: 0.76 | tod loss: 0.23
* Batch 20/365 - fog loss: 0.10 | glare loss: 0.34 | road loss: 0.37 | traffic loss: 0.21 | weather loss: 0.75 | scene loss: 0.78 | tod loss: 0.28
* Batch 30/365 - fog loss: 0.00 | glare loss: 0.33 | road loss: 0.37 | traffic loss: 0.21 | weather loss: 0.64 | scene loss: 0.68 | tod loss: 0.19
* Batch 40/365 - fog loss: 0.07 | glare loss: 0.28 | road loss: 0.33 | traffic loss: 0.18 | weather loss: 0.70 | scene loss: 0.66 | tod loss: 0.29
* Batch 50/365 - fog loss: 0.07 | glare loss: 0.28 | road loss: 0.28 | traffic loss: 0.17 | weather loss: 0.61 | scene loss: 0.67 | tod loss: 0.18
* Batch 60/365 - fog loss: 0.05 | glare loss: 0.28 | road loss: 0.35 | traffic loss: 0.18 | weather loss: 0.67 | scene loss: 0.68 | tod loss: 0.20
* Batch 70/365 - fog loss: 0.05 | glare loss: 0.27 | road loss: 0.43 | traffic loss: 0.20 | 

# (Optional) Test Evaluation

In [13]:
# Set writer to None to disable TensorBoard logging
trainer.evaluate_model(model, testloader, device, writer=writer)

[LOSSES] - fog loss: 0.10 | glare loss: 1.05 | road loss: 0.93 | traffic loss: 0.52 | weather loss: 1.77 | scene loss: 2.08 | tod loss: 0.73
[ACCURACIES] - fog accuracy: 99.22% | glare accuracy: 88.85% | road accuracy: 89.98% | traffic accuracy: 94.29% | weather accuracy: 80.77% | scene accuracy: 76.14% | tod accuracy: 93.34%
OVERALL LOSS - 7.17
OVERALL ACCURACY - 88.94%


# Visualize Results

First, ensure you're in the correct environment:

`conda env create -f environment.yml`

This will create a conda environment called *tensorboard*, which you can activate via `conda activate tensorboard`

Then, to visualize the results: 

`tensorboard --logdir=runs`

This will automatically and recursively scan through all run logs in the runs/ directory.